In [1]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS'

In [4]:
import tensorflow as tf

def build_model(input_shape, classes, learning_rate):
    """
    Builds a simple CNN model. 
    You can easily swap this for Transfer Learning (VGG16) later.
    """
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_classes: int

In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training
        params = self.params

        create_directories([config.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(config.root_dir),
            trained_model_path= Path(config.trained_model_path),
            training_data= Path(config.training_data),
            params_epochs= params.EPOCHS,
            params_batch_size= params.BATCH_SIZE,
            params_is_augmentation= params.AUGMENTATION,
            params_image_size= params.IMAGE_SIZE,
            params_classes= params.CLASSES    
        )

        return training_config

     

In [14]:
import tensorflow as tf
from pathlib import Path
from cnnClassifier import logger



class ModelTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def train(self):
        # 1. Load the transformed datasets
        train_ds = tf.data.Dataset.load(str(self.config.training_data))
        
        # 2. Build the model from the architecture file
        model = build_model(
            input_shape=tuple(self.config.params_image_size),
            classes=self.config.params_classes,
            learning_rate=0.001
        )
        
        # 3. Train
        model.fit(
            train_ds,
            epochs=self.config.params_epochs
        )
        
        # 4. Save model
        model.save(self.config.trained_model_path)

In [15]:
STAGE_NAME = "Model Training Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    training_config = config.get_training_config()
    model_trainer = ModelTrainer(config=training_config)
    model_trainer.train()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-27 23:02:04,291: INFO: 1057139548: >>>>>> stage Model Training Stage started <<<<<<]
[2026-06-27 23:02:04,307: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 23:02:04,310: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 23:02:04,312: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-27 23:02:04,314: INFO: common: created directory at: artifacts]
[2026-06-27 23:02:04,315: INFO: common: created directory at: artifacts/training]


d:\Revision\Covid-19_detection_with_MLOPS\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[2026-06-27 23:02:04,718: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]


d:\Revision\Covid-19_detection_with_MLOPS\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


691/691 ━━━━━━━━━━━━━━━━━━━━ 250s 359ms/step - accuracy: 0.8586 - loss: 0.4616
[2026-06-27 23:06:15,040: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
[2026-06-27 23:06:16,070: INFO: 1057139548: >>>>>> stage Model Training Stage completed <<<<<<

x==========x]
